# Week 96, Day 3: Contribution sprint

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w096d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('csvline.py', 'w', encoding='utf-8') as f:
    f.write('def split_fields(line, sep=","):\n    """Split one CSV line into fields, honouring double quotes."""\n    if not line:\n        return []\n    fields, current, quoted = [], [], False\n    i = 0\n    while i < len(line):\n        ch = line[i]\n        if quoted:\n            if ch == \'"\':\n                if i + 1 < len(line) and line[i + 1] == \'"\':\n                    current.append(\'"\')\n                    i += 1\n                else:\n                    quoted = False\n            else:\n                current.append(ch)\n        elif ch == \'"\' and not current:\n            quoted = True\n        elif ch == sep:\n            fields.append("".join(current))\n            current = []\n        else:\n            current.append(ch)\n        i += 1\n    if current:\n        fields.append("".join(current))\n    return fields\n')

**Predict before you run:** what will this print?



In [ ]:
import csv
from csvline import split_fields

for line in ["a,b,", 'a,""', '"x""y",z']:
    print(split_fields(line), next(csv.reader([line])))

> **Key idea.** Use a trusted implementation as an oracle (differential testing); treat differences as leads. Sprint in order: reproduce, shrink, find the cause, fix minimally, test, fuzz, self-review, open. Seed fuzzers so failures reproduce; for small alphabets, check every input up to a length, before and after the fix.

### Exercise: Comparing with the oracle

Write `csv_oracle(line)` returning the fields `csv.reader` gives for one line (`[]` if it gives no row), and `differences(fn, lines)`: `(line, OURS, ORACLE)` for each line where `fn(line)` differs.

In [ ]:
import csv

def csv_oracle(line):
    ___

def differences(fn, lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from csvline import split_fields\n\ndef test_differences():\n    \"the report, and a little more\"\n    assert differences(split_fields, [\"a,b,\", 'a,\"\"', '\"x\"\"y\",z', \"\", \"a,b\"]) == [\n        (\"a,b,\", [\"a\", \"b\"], [\"a\", \"b\", \"\"]), ('a,\"\"', [\"a\"], [\"a\", \"\"])]\n    assert csv_oracle(\"\") == [] and csv_oracle(\",\") == [\"\", \"\"]")

### Exercise: Shrinking a string

Write `shrink(text, fails)`: repeatedly try deleting one character at a time (from the first); keep the first deletion that still fails (an empty string counts if it fails) and start again; stop when no single deletion fails.

In [ ]:
def shrink(text, fails):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import csv\nfrom csvline import split_fields\n\ndef test_shrink():\n    \"to a single character\"\n    oracle = lambda l: (list(csv.reader([l])) or [[]])[0]\n    fails = lambda l: split_fields(l) != oracle(l)\n    assert shrink(\"walk,Mill to Ridge,\", fails) == \",\"\n    assert shrink('a,\"\"', fails) == '\"'\n    assert shrink(\"abc\", lambda t: \"b\" in t) == \"b\"")

### Exercise: The fix

Write the corrected `split_fields(line, sep=",")`: an empty line gives `[]`; otherwise split on `sep`; a field starting with `"` is quoted, where `sep` is ordinary, `""` means one `"`, and the next `"` ends the quoting (later characters still belong to the field); a `"` elsewhere is an ordinary character; and the last field is always kept, even if it's empty.

In [ ]:
def split_fields(line, sep=","):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import csv, itertools\n\ndef test_cases():\n    \"the report\"\n    assert split_fields(\"a,b,\") == [\"a\", \"b\", \"\"] and split_fields('a,\"\"') == [\"a\", \"\"]\n    assert split_fields('\"x\"\"y\",z') == ['x\"y', \"z\"] and split_fields('\"a,b\",c') == [\"a,b\", \"c\"]\n    assert split_fields(\"\") == [] and split_fields(\",\") == [\"\", \"\"] and split_fields('a\"b') == ['a\"b']\n\ndef test_against_csv():\n    \"every short line agrees with the oracle\"\n    for n in range(6):\n        for t in itertools.product(',\"a ', repeat=n):\n            line = \"\".join(t)\n            rows = list(csv.reader([line]))\n            assert split_fields(line) == (rows[0] if rows else []), repr(line)")

### Exercise: Tests for the fix

Write a `unittest` suite for `split_fields` that passes on the fixed version and catches all five broken ones: the original bug, doubled quotes kept as two, quotes treated as special in the middle of a field, an empty line giving one empty field, and the separator dropped inside quotes.

In [ ]:
import unittest
from csvline import split_fields

class TestSplitFields(unittest.TestCase):
    def test_simple(self):
        self.assertEqual(split_fields("a,b"), ["a", "b"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BUGGY = open(\"csvline.py\").read()\nGOOD = BUGGY.replace('    if current:\\n        fields.append(\"\".join(current))', '    fields.append(\"\".join(current))')\nBROKEN = {\n    \"the original bug\": BUGGY,\n    \"doubled quotes kept\": GOOD.replace(\"current.append('\\\"')\\n                    i += 1\", \"current.append('\\\"')\"),\n    \"quotes anywhere\": GOOD.replace(\"elif ch == '\\\"' and not current:\", \"elif ch == '\\\"':\"),\n    \"empty line gives a field\": GOOD.replace(\"    if not line:\\n        return []\\n\", \"\"),\n    \"separator ignored in quotes\": GOOD.replace(\"            else:\\n                current.append(ch)\\n        elif\", \"            elif ch != sep:\\n                current.append(ch)\\n        elif\"),\n}\n\ndef test_suite():\n    \"all five caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    write_file(\"csvline.py\", GOOD)\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"csvline\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"csvline.py\", BUGGY)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: A seeded fuzzer

Write `fuzz(fn, oracle, alphabet, count, max_len, seed)`: with `random.Random(seed)`, make `count` strings, each of a length chosen with `rng.randint(0, max_len)` and characters chosen with `rng.choice(alphabet)`; return the distinct strings (in order of first appearance) on which `fn` and `oracle` disagree.

In [ ]:
import random

def fuzz(fn, oracle, alphabet, count, max_len, seed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import csv\nfrom csvline import split_fields\n\nORACLE = lambda l: (list(csv.reader([l])) or [[]])[0]\n\ndef test_fuzz():\n    \"random, but reproducible\"\n    found = fuzz(split_fields, ORACLE, ',\"a ', 500, 8, 1)\n    assert found and found == fuzz(split_fields, ORACLE, ',\"a ', 500, 8, 1)\n    assert len(found) == len(set(found)) and all(split_fields(l) != ORACLE(l) for l in found)\n    fixed = lambda l: ORACLE(l)\n    assert fuzz(fixed, ORACLE, ',\"a ', 200, 8, 2) == []")

### Bug hunt: A failure nobody could reproduce

The fuzzer uses an unseeded generator, so every run tries different strings, and two runs rarely agree: this prints `False`. Seed it so the same run gives the same strings, and it prints `True`.

In [ ]:
import random

def strings(count):
    rng = random.Random()
    return ["".join(rng.choice(',"a ') for _ in range(6)) for _ in range(count)]

print(strings(20) == strings(20))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a seed\"\n    expect_output(\"True\")")

### Bug hunt: Exhaustive, except for the short ones

`range(1, max_len)` misses both the empty string and the longest strings, so only 14 of the 31 strings up to length 4 over two characters are checked, and this prints `14`. Check every length from 0 to `max_len` so it prints `31`.

In [ ]:
import itertools

max_len = 4
inputs = ["".join(t) for n in range(1, max_len) for t in itertools.product("a,", repeat=n)]
print(len(inputs))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"0 to max_len\"\n    expect_output(\"31\")")

### Puzzle: Every short input

Arrange the exhaustive check: every string over the alphabet up to `max_len`, compared with the oracle. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # next(csv.reader(["a,b,"]))
q = ___   # how many strings of length 0 to 2 there are over 4 characters
r = ___   # the testing technique that compares two implementations (two words, lower case)
s = ___   # True or False: a difference from the oracle always means our code is wrong
t = ___   # split_fields('"x""y"') with the fix (a list)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"a\", \"b\", \"\"] and q == 21 and r == \"differential testing\", \"p, q and r\"\n    assert s is False and t == ['x\"y'], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: before and after, exhaustively

Write `sprint_report(buggy_source, fixed_source, alphabet, max_len)`. Load `split_fields` from each source; check every string over `alphabet` of length 0 to `max_len` against `csv` as the oracle. Return `(lines, verdict)`: `"checked: N inputs"`; for `before` and `after`, `"LABEL: D differ, smallest S"` (S the shortest differing string, ties alphabetical, with `repr`) or `"LABEL: 0 differ"`; then `"diff: +A -R"` from `difflib.unified_diff` of the two sources' lines. `verdict` is `"fixed"` if nothing differs after and A + R is at most 4, else `"not yet"`.

In [ ]:
import csv
import difflib
import itertools

def sprint_report(buggy_source, fixed_source, alphabet, max_len):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BUGGY = open(\"csvline.py\").read()\nFIXED = BUGGY.replace('    if current:\\n        fields.append(\"\".join(current))', '    fields.append(\"\".join(current))')\n\ndef test_fixed():\n    \"every input up to four characters\"\n    assert sprint_report(BUGGY, FIXED, ',\"a ', 4) == (\n        [\"checked: 341 inputs\", \"before: 86 differ, smallest '\\\"'\", \"after: 0 differ\", \"diff: +1 -2\"], \"fixed\")\n\ndef test_not_yet():\n    \"no change\"\n    assert sprint_report(BUGGY, BUGGY, ',\"a', 3) == (\n        [\"checked: 40 inputs\", \"before: 15 differ, smallest '\\\"'\", \"after: 15 differ, smallest '\\\"'\", \"diff: +0 -0\"], \"not yet\")")

## Warm-up

The first two disagree with `csv`: a field that's empty at the end of the line disappears. The third agrees: doubled quotes inside quotes already work. Two minutes, and the bug is reproduced and bounded.

## An oracle for free

When a trusted implementation of the same thing exists (the standard library, an older version, a slower but obviously correct version, another language's library), you don't need to work out every expected answer by hand: compare. **Differential testing** has found thousands of bugs in compilers, parsers and databases this way. Its limits: the oracle can have bugs too, and the two may differ on purpose (decide each difference on its merits), so a difference is a lead, not a verdict.

## Fast, in order

Sprint speed comes from doing the steps in order without skipping, not from rushing any of them:

1. **Reproduce** against the oracle (two minutes).
2. **Shrink**: delete characters one at a time while it still differs, to `,` or `""`.
3. **Cause**: the final `if current:` only keeps the last field if it has characters, but an empty last field is still a field.
4. **Fix**: append the last field unconditionally (one line removed, one changed).
5. **Test**: the regression test, plus tests that would catch other plausible mistakes.
6. **Fuzz and check exhaustively** (below), then self-review and open the pull request.

## Random and exhaustive

**Fuzzing** throws random inputs at the code: here, random strings of commas, quotes, letters and spaces, compared with the oracle. Seed the random generator (`random.Random(1)`) so a failure can be reproduced exactly. For small alphabets, you can go further and check **every** string up to a length: four characters and lengths up to 6 is about 5,500 strings, a fraction of a second. If the fixed version agrees with the oracle on all of them, you can be much more confident than a handful of examples could make you. Hypothesis (on your own computer) combines the two, and shrinks the failures for you.

## Practice

## Quiz

## Daily challenge

## Your sprint

1. **Pick a bug** with an obvious oracle if you can: a function that should agree with another library, an older version, or a simple slow version you can write in ten lines.
2. **Run the whole rhythm**: reproduce, shrink, cause, fix, test, fuzz (seeded), and an exhaustive check if the inputs allow.
3. **Open the pull request** with the oracle comparison mentioned in the description, and log it.

## Before you go

Tomorrow you'll be on the other side: reviewing other people's pull requests, which is one of the most valuable things a contributor can do.